In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time



# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    step3: str


# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt ")
    time.sleep(30) # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"step3": "done"}


# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.add_edge(START, "step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)


In [3]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": "thread-1"}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")



▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt 
✅ Step 3 executed


In [4]:

graph.get_state({"configurable": {"thread_id": "thread-1"}})


StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f196d80-be5d-6d3a-8003-f8fc9f84b08b'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-08-13T05:30:08.191192+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f196d80-be5a-61d7-8002-810ee7db1348'}}, tasks=(), interrupts=())

In [5]:

list(graph.get_state_history({"configurable": {"thread_id": "thread-1"}}))


[StateSnapshot(values={'input': 'start', 'step1': 'done'}, next=('step_2',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f196d4e-5953-6398-8001-64bffda5337b'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-08-13T05:07:35.418953+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f196d4e-594e-6e8d-8000-9d17a50e81a1'}}, tasks=(PregelTask(id='4ce68902-270b-5122-7473-77d9c3e7ac5c', name='step_2', path=('__pregel_pull', 'step_2'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'input': 'start'}, next=('step_1',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f196d4e-594e-6e8d-8000-9d17a50e81a1'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-08-13T05:07:35.417180+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'ch

In [6]:
#resuming after crash
graph.invoke(None, config={"configurable": {"thread_id": "thread-1"}})

⏳ Step 2 hanging... now manually interrupt 
✅ Step 3 executed


{'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}

In [7]:

graph.get_state({"configurable": {"thread_id": "thread-1"}})


StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f196d52-ccc4-69ff-8003-4121b5ae44e6'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-08-13T05:09:34.898211+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f196d52-ccc1-6580-8002-5d1daed72934'}}, tasks=(), interrupts=())

In [8]:

list(graph.get_state_history({"configurable": {"thread_id": "thread-1"}}))


[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f196d52-ccc4-69ff-8003-4121b5ae44e6'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-08-13T05:09:34.898211+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f196d52-ccc1-6580-8002-5d1daed72934'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f196d52-ccc1-6580-8002-5d1daed72934'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-13T05:09:34.896853+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f196d4e-5953-6398-8001-64bffda5337b'}}, tasks=(PregelTask(id='fb3aa7e6-03fe-0a4c